In [1]:
%load_ext autoreload
%autoreload 2

import torch
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetbot import bgr8_to_jpeg

# 사용자 정의 모듈 임포트
from modules.hardware import AGVHardware
from modules.driving_logic import LineTrackingBrain
from modules.mission_manager import MissionManager

# 1. 주행 모델 로드
# (경로는 실제 모델 파일 위치로 수정해주세요)
model_path = 'best_steering_model_xy.pth' 
try:
    model = torch.load(model_path)
    device = torch.device('cuda')
    model = model.to(device)
    model.eval().half()
    print(f"✅ 모델 로드 완료: {model_path}")
except FileNotFoundError:
    print(f"❌ 모델 파일을 찾을 수 없습니다: {model_path}")
    print("   -> 주행 기능이 작동하지 않을 수 있습니다.")
    model = None

ModuleNotFoundError: No module named 'torch'

In [ ]:
# 2. 어안 렌즈 캘리브레이션 데이터 로드
try:
    calib_data = np.load('fisheye_calib.npz')
    K = calib_data['K']
    D = calib_data['D']
    # 맵 생성 (해상도 224x224 기준)
    # balance 값을 조절하여 검은 테두리 영역 조절 가능 (0.0 ~ 1.0)
    new_K = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(K, D, (224, 224), np.eye(3), balance=0.5)
    map1, map2 = cv2.fisheye.initUndistortRectifyMap(K, D, np.eye(3), new_K, (224, 224), cv2.CV_16SC2)
    print("✅ 캘리브레이션 데이터 로드 완료")
except Exception as e:
    print("⚠️ 캘리브레이션 파일 로드 실패 (OCR 품질이 떨어질 수 있음)")
    print(f"   Error: {e}")
    map1, map2 = None, None

# 3. OCR 감지기 준비 (Mock 또는 실제 객체)
class MockOCR:
    """테스트용 가짜 OCR 감지기"""
    def detect(self, image, target=None):
        print(f"[MockOCR] 이미지 분석 중... (Target: {target})")
        # 테스트를 위해 항상 성공한다고 가정하고 더미 결과 반환
        class Result:
            text = "187고1604"
        return Result()

# 실제 사용할 때는 ClovaOCR 객체 등으로 교체하세요
ocr_detector = MockOCR()

# 4. 전체 시스템 조립
agv = AGVHardware()
brain = LineTrackingBrain(model, device) if model else None
manager = MissionManager(agv, brain, ocr_detector, map1, map2)

print("🚀 시스템 준비 완료! (Ready to Tracking)")

In [ ]:
# --- 제어 위젯 ---
speed_slider = widgets.FloatSlider(value=0.15, min=0.0, max=0.5, step=0.01, description='Speed')
steering_gain = widgets.FloatSlider(value=0.04, min=0.0, max=0.2, step=0.001, description='St. Gain')
steering_dgain = widgets.FloatSlider(value=0.0, min=0.0, max=0.5, step=0.001, description='St. DGain')
steering_bias = widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.01, description='Bias')

# --- 모니터링 위젯 ---
image_widget = widgets.Image(format='jpeg', width=224, height=224)
x_slider = widgets.FloatSlider(min=-1.0, max=1.0, description='Detected X')
y_slider = widgets.FloatSlider(min=0, max=1.0, orientation='vertical', description='Detected Y')
state_label = widgets.Label(value="Current State: IDLE")
ocr_label = widgets.Label(value="OCR Result: None")

# --- 버튼 ---
start_btn = widgets.Button(description='Start Mission', button_style='success')
stop_btn = widgets.Button(description='Stop', button_style='danger')

def on_start(_): 
    # 미션 초기화 (플래그 리셋 등 필요 시 context 접근)
    manager.context.is_mission_completed = False
    manager.set_state("TRACKING")

def on_stop(_): 
    manager.set_state("IDLE")

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)

# --- 레이아웃 ---
control_box = widgets.VBox([speed_slider, steering_gain, steering_dgain, steering_bias])
info_box = widgets.VBox([state_label, ocr_label, start_btn, stop_btn])
display(widgets.HBox([image_widget, y_slider, control_box, info_box]), x_slider)

In [ ]:
import time

try:
    while True:
        # 1. UI 값 -> Context 반영
        ctx = manager.context
        ctx.speed_gain = speed_slider.value
        ctx.steering_gain = steering_gain.value
        ctx.steering_dgain = steering_dgain.value
        ctx.steering_bias = steering_bias.value
        
        # 2. 미션 매니저 업데이트 (상태별 로직 수행)
        manager.update()
        
        # 3. UI 업데이트 (Context -> UI)
        if ctx.processed_image: # 주행 중일 때
            image_widget.value = ctx.processed_image
        elif manager.hw.camera: # 주행 아닐 때도 카메라 화면 갱신
            frame = manager.hw.get_frame()
            if frame is not None:
                image_widget.value = bgr8_to_jpeg(frame)

        x_slider.value = ctx.current_x
        y_slider.value = ctx.current_y
        
        # 상태 및 OCR 결과 표시
        state_label.value = f"Current State: {manager.current_state_name}"
        if ctx.ocr_result:
            ocr_label.value = f"OCR Result: {ctx.ocr_result}"
            
        time.sleep(0.01)
        
except KeyboardInterrupt:
    manager.set_state("IDLE")
    print("프로그램이 종료되었습니다.")